# 01 — Data Quality Audit & Rules Catalog

> **Mục tiêu**: Đánh giá toàn diện 7 chiều chất lượng dữ liệu (Completeness, Validity, Consistency, Uniqueness, Accuracy, Timeliness, Referential Integrity), rà soát rủi ro Data Leakage, phân loại Missing Values theo bản chất nghiệp vụ và thiết lập Data Quality Rules Catalog phục vụ cả Model Training lẫn Real-time Scoring trên SAS Fraud Decisioning.
>
> **Input**: 11 bảng normalized CSV trong `fraud_data_generator_v2/output_training_raw/merged/`.
>
> **Output**: Data Profiling Report, Data Quality Issues Log, Missing Value Strategy, Leakage Audit và Runtime DQ Monitoring Plan.

---

## Table of Contents

1. [Thiết lập Môi trường và Nạp Dữ liệu](#1-thiết-lập-môi-trường-và-nạp-dữ-liệu)
2. [Data Profiling & Khảo sát Cấu trúc Tổng thể](#2-data-profiling--khảo-sát-cấu-trúc-tổng-thể)
3. [Kiểm tra Tính duy nhất & Toàn vẹn Khóa (Uniqueness & Referential Integrity)](#3-kiểm-tra-tính-duy-nhất--toàn-vẹn-khóa-uniqueness--referential-integrity)
4. [Kiểm tra Tính hợp lệ & Miền giá trị (Validity & Value Range Checks)](#4-kiểm-tra-tính-hợp-lệ--miền-giá-trị-validity--value-range-checks)
5. [Phân tích & Phân loại Giá trị Thiếu (Completeness & Missing Value Taxonomy)](#5-phân-tích--phân-loại-giá-trị-thiếu-completeness--missing-value-taxonomy)
6. [Kiểm tra Tính nhất quán & Chuỗi Nghiệp vụ (Consistency & Business Invariants)](#6-kiểm-tra-tính-nhất-quán--chuỗi-nghiệp-vụ-consistency--business-invariants)
7. [Kiểm tra Tính nhân quả Thời gian (Timeliness & Causality Checks)](#7-kiểm-tra-tính-nhân-quả-thời-gian-timeliness--causality-checks)
8. [Kiểm tra Độ ổn định Đa nguồn (Cross-Run Distribution Stability)](#8-kiểm-tra-độ-ổn-định-đa-nguồn-cross-run-distribution-stability)
9. [Rà soát Rủi ro Rò rỉ Dữ liệu (Data Leakage & Shortcut Audit)](#9-rà-soát-rủi-ro-rò-rỉ-dữ-liệu-data-leakage--shortcut-audit)
10. [Tổng kết: Data Quality Rules Catalog & Kế hoạch Giám sát Production](#10-tổng-kết-data-quality-rules-catalog--kế-hoạch-giám-sát-production)


---
## 1. Thiết lập Môi trường và Nạp Dữ liệu

### 1.1 Khởi tạo Môi trường và Khung ghi nhận Data Quality Log

- **Mục tiêu**: Khởi tạo thư viện phân tích, cấu hình định dạng hiển thị bảng biểu, thiết lập đường dẫn tới thư mục dữ liệu hợp nhất (`merged/`) và khai báo hàm helper `log_dq` để tự động tích lũy kết quả kiểm định chất lượng xuyên suốt notebook.
- **Input**: Đường dẫn thư mục `../fraud_data_generator_v2/output_training_raw/merged/`.
- **Output kỳ vọng**: Môi trường sẵn sàng, hàm `log_dq` khởi tạo thành công với cấu trúc log 6 trường chuẩn: `Check | Table / Column | Actual | Expected | Status | Action`.


In [45]:
import json
import warnings
from pathlib import Path
import pandas as pd
import numpy as np

# Thiết lập hiển thị pandas
warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 100)
pd.set_option("display.max_rows", 100)
pd.set_option("display.float_format", "{:,.4f}".format)

# Thiết lập đường dẫn dữ liệu
DATA_DIR = Path(r"../fraud_data_generator_v2/output_training_raw/merged")
assert DATA_DIR.exists(), f"Không tìm thấy thư mục dữ liệu tại: {DATA_DIR.resolve()}"

# Danh sách log theo dõi Data Quality Issues xuyên suốt notebook
dq_issues = []

def log_dq(check_name: str, table_col: str, actual: str, expected: str, status: str, action: str):
    """
    Ghi nhận một kết quả kiểm tra chất lượng dữ liệu.
    Status hợp lệ: PASS (Đạt) | WARN (Cảnh báo nghiệp vụ) | FAIL (Lỗi nghiêm trọng).
    """
    assert status in ["PASS", "WARN", "FAIL"], f"Status không hợp lệ: {status}"
    dq_issues.append({
        "Check": check_name,
        "Table / Column": table_col,
        "Actual": str(actual),
        "Expected": str(expected),
        "Status": status,
        "Action": action
    })

print("✓ Khởi tạo môi trường và khung ghi nhận DQ Log thành công.")


✓ Khởi tạo môi trường và khung ghi nhận DQ Log thành công.


### 1.2 Nạp 11 Bảng Dữ liệu Hợp nhất

- **Mục tiêu**: Đọc toàn bộ 11 bảng dữ liệu normalized CSV được sinh từ 5 simulation runs độc lập (`RUN_TXN_TRAIN_001` → `005`). Tất cả các cột được đọc ban đầu dưới dạng chuỗi (`dtype=str`) để kiểm tra nguyên trạng định dạng gốc trước khi ép kiểu.
- **Input**: 11 file CSV trong thư mục `merged/`.
- **Output kỳ vọng**: 11 DataFrames được nạp thành công, hiển thị bảng tóm tắt số dòng, số cột của từng bảng.


In [46]:
# Nạp 11 bảng dữ liệu liên quan đến Transaction Fraud Domain
TABLE_NAMES = [
    "transactions",
    "transaction_features",
    "scenario_event_entities",
    "customers",
    "accounts",
    "devices",
    "login_sessions",
    "beneficiaries",
    "account_change_events",
    "auth_events",
    "fraud_ground_truth"
]

tables = {}
for name in TABLE_NAMES:
    file_path = DATA_DIR / f"{name}.csv"
    assert file_path.exists(), f"Thiếu file: {file_path}"
    tables[name] = pd.read_csv(file_path, dtype=str)

# Hiển thị bảng tổng quan quy mô dữ liệu
summary_rows = []
for name, df in tables.items():
    summary_rows.append({
        "Bảng dữ liệu": name,
        "Số dòng": f"{len(df):,}",
        "Số cột": len(df.columns),
        "Dung lượng ước tính (MB)": f"{df.memory_usage(deep=True).sum() / 1024**2:.2f}"
    })

summary_df = pd.DataFrame(summary_rows)
print("=== TỔNG QUAN QUY MÔ DỮ LIỆU ĐÃ NẠP (5 SIMULATION RUNS MERGED) ===")
display(summary_df)


=== TỔNG QUAN QUY MÔ DỮ LIỆU ĐÃ NẠP (5 SIMULATION RUNS MERGED) ===


,Bảng dữ liệu,Số dòng,Số cột,Dung lượng ước tính (MB)
0,transactions,"111,064",28,203.05
1,transaction_features,"111,064",14,106.31
2,scenario_event_entities,"4,095",11,3.08
3,customers,"7,500",23,12.53
4,accounts,"9,651",16,10.58
5,devices,"10,625",12,8.93
6,login_sessions,"26,676",21,37.11
7,beneficiaries,"20,467",12,17.27
8,account_change_events,"2,573",13,2.60
9,auth_events,"39,693",12,30.64


---
## 2. Data Profiling & Khảo sát Cấu trúc Tổng thể

### 2.1 Đối chiếu Schema thực tế với Data Contract Chuẩn

- **Mục tiêu**: Kiểm tra tính đầy đủ và chính xác của tên cột trên các bảng trọng yếu (`transactions`, `customers`, `accounts`, `scenario_event_entities`) so với hợp đồng dữ liệu chuẩn (`generators/engine.py`). Phát hiện các trường hợp cột bị đổi tên, cột bị thiếu hoặc cột lạ ngoài thiết kế.
- **Input**: Danh sách cột thực tế của từng bảng so với `EXPECTED_SCHEMA`.
- **Output kỳ vọng**: 100% các cột khớp chính xác với thiết kế, không thiếu cột nghiệp vụ, không thừa cột không xác định (`PASS`).


In [47]:
# Schema chuẩn được quy định trong Data Contract (generators/engine.py - M)
EXPECTED_SCHEMA = {
    "transactions": [
        "transaction_id", "simulation_run_id", "account_id", "customer_id", "session_id", "device_id", 
        "beneficiary_id", "transaction_at", "amount", "currency", "direction", "transaction_type", 
        "channel", "counterparty_account_hash", "counterparty_bank", "counterparty_internal_account_id", 
        "merchant_id", "merchant_category_code", "ip_address", "province", "country", "vpn_flag", 
        "proxy_flag", "status", "failure_reason", "balance_before", "balance_after", "created_at"
    ],
    "customers": [
        "customer_id", "simulation_run_id", "customer_type", "full_name", "gender", "dob", 
        "id_number_hash", "phone_hash", "email_hash", "province", "address_cluster_id", "phone_cluster_id", 
        "occupation_group", "income_band", "customer_segment", "onboarding_channel", "onboarding_date", 
        "kyc_level", "base_risk_level", "customer_status", "is_synthetic_identity_seed", "is_mule_candidate_seed", "created_at"
    ],
    "accounts": [
        "account_id", "simulation_run_id", "customer_id", "account_no_hash", "account_type", 
        "account_currency", "open_date", "status", "branch_code", "account_opening_channel", 
        "home_province", "daily_transfer_limit", "single_txn_limit", "average_balance", "dormant_since", "created_at"
    ],
    "scenario_event_entities": [
        "event_id", "scenario_code", "entity_type", "entity_id", "entity_role", 
        "label_scope", "target_fraud", "hard_negative", "sample_weight", "valid_from", "valid_to"
    ]
}

schema_results = []
for tbl, exp_cols in EXPECTED_SCHEMA.items():
    actual_cols = list(tables[tbl].columns)
    missing_cols = set(exp_cols) - set(actual_cols)
    extra_cols = set(actual_cols) - set(exp_cols)
    
    is_match = (missing_cols == set() and extra_cols == set())
    status = "PASS" if is_match else "FAIL"
    
    schema_results.append({
        "Bảng": tbl,
        "Số cột chuẩn": len(exp_cols),
        "Số cột thực tế": len(actual_cols),
        "Cột thiếu": list(missing_cols) if missing_cols else "Không",
        "Cột thừa": list(extra_cols) if extra_cols else "Không",
        "Trạng thái": status
    })
    
    log_dq(
        check_name="Schema & Header Match",
        table_col=tbl,
        actual=f"{len(actual_cols)} cols (extra: {len(extra_cols)}, miss: {len(missing_cols)})",
        expected=f"{len(exp_cols)} cols exactly",
        status=status,
        action="Giữ nguyên schema" if status == "PASS" else "Cần update lại generator mapping"
    )

display(pd.DataFrame(schema_results))


,Bảng,Số cột chuẩn,Số cột thực tế,Cột thiếu,Cột thừa,Trạng thái
0,transactions,28,28,Không,Không,PASS
1,customers,23,23,Không,Không,PASS
2,accounts,16,16,Không,Không,PASS
3,scenario_event_entities,11,11,Không,Không,PASS


### 2.2 Khảo sát Kiểu dữ liệu và Khả năng ép kiểu (Type Casting Audit)

- **Mục tiêu**: Kiểm tra xem các trường số liệu (`amount`, `balance_before`, `balance_after`, `limits`), trường thời gian (`transaction_at`, `login_at`, `open_date`) và trường boolean (`vpn_flag`, `is_new_device`, `is_emulator`) có thể ép kiểu an toàn sang numeric/datetime/boolean mà không bị lỗi parse hay không.
- **Input**: DataFrame `transactions`, `login_sessions`, `devices`, `accounts`.
- **Output kỳ vọng**: Số lượng bản ghi bị lỗi parse khi ép kiểu = 0 đối với các cột bắt buộc.


In [48]:
# Kiểm tra khả năng parse của các nhóm trường đặc thù
type_checks = []

# 1. Parse Numeric trên bảng transactions
for num_col in ["amount", "balance_before", "balance_after"]:
    series = tables["transactions"][num_col]
    parsed = pd.to_numeric(series, errors="coerce")
    failed_count = parsed.isna().sum() - series.isna().sum()
    status = "PASS" if failed_count == 0 else "FAIL"
    type_checks.append({
        "Bảng Cột": f"transactions.{num_col}",
        "Kiểu mong đợi": "Float / Numeric",
        "Lỗi parse (coerce nulls)": failed_count,
        "Trạng thái": status
    })

# 2. Parse Datetime trên transactions & login_sessions
for tbl, dt_col in [("transactions", "transaction_at"), ("login_sessions", "login_at"), ("accounts", "open_date")]:
    series = tables[tbl][dt_col]
    parsed = pd.to_datetime(series, errors="coerce", utc=True)
    failed_count = parsed.isna().sum() - series.isna().sum()
    status = "PASS" if failed_count == 0 else "FAIL"
    type_checks.append({
        "Bảng Cột": f"{tbl}.{dt_col}",
        "Kiểu mong đợi": "ISO-8601 Datetime",
        "Lỗi parse (coerce nulls)": failed_count,
        "Trạng thái": status
    })

# 3. Parse Boolean string ('true' / 'false')
for tbl, bool_col in [("transactions", "vpn_flag"), ("login_sessions", "is_new_device"), ("devices", "is_emulator")]:
    series = tables[tbl][bool_col].dropna().str.lower()
    invalid_bools = (~series.isin(["true", "false"])).sum()
    status = "PASS" if invalid_bools == 0 else "FAIL"
    type_checks.append({
        "Bảng Cột": f"{tbl}.{bool_col}",
        "Kiểu mong đợi": "Boolean ('true'/'false')",
        "Lỗi parse (coerce nulls)": invalid_bools,
        "Trạng thái": status
    })

type_df = pd.DataFrame(type_checks)
display(type_df)

for r in type_checks:
    log_dq(
        check_name="Dtype Castability",
        table_col=r["Bảng Cột"],
        actual=f"Lỗi parse: {r['Lỗi parse (coerce nulls)']}",
        expected="0 lỗi parse",
        status=r["Trạng thái"],
        action="Ép kiểu an toàn khi feature engineering" if r["Trạng thái"] == "PASS" else "Cần clean chuỗi trước khi ép kiểu"
    )


,Bảng Cột,Kiểu mong đợi,Lỗi parse (coerce nulls),Trạng thái
0,transactions.amount,Float / Numeric,0,PASS
1,transactions.balance_before,Float / Numeric,0,PASS
2,transactions.balance_after,Float / Numeric,0,PASS
3,transactions.transaction_at,ISO-8601 Datetime,0,PASS
4,login_sessions.login_at,ISO-8601 Datetime,0,PASS
5,accounts.open_date,ISO-8601 Datetime,0,PASS
6,transactions.vpn_flag,Boolean ('true'/'false'),0,PASS
7,login_sessions.is_new_device,Boolean ('true'/'false'),0,PASS
8,devices.is_emulator,Boolean ('true'/'false'),0,PASS


### 2.3 Rà soát Cột Đơn trị (Constant Columns) & Cardinality Cực thấp

- **Mục tiêu**: Phát hiện các cột chỉ có đúng 1 giá trị duy nhất (Zero-variance / Constant columns) hoặc gần như đơn trị. Các cột này hoàn toàn không có giá trị phân biệt trong mô hình ML và gây lãng phí tài nguyên tính toán/truy vấn.
- **Input**: Toàn bộ 28 cột của bảng `transactions`.
- **Output kỳ vọng**: Bảng thống kê số lượng giá trị duy nhất (`nunique`) của từng cột, xác định rõ các cột cần drop trước khi đưa vào pipeline.


In [49]:
txn_df = tables["transactions"]
cardinality_data = []

for col in txn_df.columns:
    unique_vals = txn_df[col].dropna().unique()
    n_unique = len(unique_vals)
    sample_val = str(unique_vals[0]) if n_unique > 0 else "NULL"
    if n_unique > 3:
        sample_val = f"{unique_vals[:2]} ... (total {n_unique})"
    
    is_constant = (n_unique <= 1)
    
    cardinality_data.append({
        "Cột": col,
        "Số giá trị duy nhất": n_unique,
        "Giá trị mẫu / Phân phối": sample_val,
        "Là cột hằng số (Constant)?": "CÓ (Cần DROP)" if is_constant else "Không"
    })

cardinality_df = pd.DataFrame(cardinality_data)
constants = cardinality_df[cardinality_df["Là cột hằng số (Constant)?"] == "CÓ (Cần DROP)"]

print("=== DANH SÁCH CÁC CỘT HẰNG SỐ (ZERO-VARIANCE) PHÁT HIỆN TRONG TRANSACTIONS ===")
display(constants)

# Ghi nhận DQ Log cho các cột hằng số
for _, row in constants.iterrows():
    log_dq(
        check_name="Constant Column Audit",
        table_col=f"transactions.{row['Cột']}",
        actual=f"1 giá trị duy nhất: {row['Giá trị mẫu / Phân phối']}",
        expected="Đa giá trị phân biệt",
        status="WARN",
        action=f"Cần loại bỏ cột '{row['Cột']}' khỏi feature matrix (Zero information value)"
    )


=== DANH SÁCH CÁC CỘT HẰNG SỐ (ZERO-VARIANCE) PHÁT HIỆN TRONG TRANSACTIONS ===


,Cột,Số giá trị duy nhất,Giá trị mẫu / Phân phối,Là cột hằng số (Constant)?
9,currency,1,VND,CÓ (Cần DROP)
16,merchant_id,0,NULL,CÓ (Cần DROP)
17,merchant_category_code,0,NULL,CÓ (Cần DROP)
20,country,1,VN,CÓ (Cần DROP)
22,proxy_flag,1,false,CÓ (Cần DROP)
23,status,1,success,CÓ (Cần DROP)
24,failure_reason,0,NULL,CÓ (Cần DROP)
27,created_at,1,2026-08-06T03:00:00+00:00,CÓ (Cần DROP)


#### Nhận xét & Đánh giá Chất lượng (Observations & Actionable Insights)

Từ kết quả rà soát trên bảng `transactions` (112.565 dòng), phát hiện **8/28 cột (chiếm 28.5% số cột)** là cột hằng số (Zero-Variance) hoặc hoàn toàn rỗng:

1. **Nhóm thông tin cố định theo phạm vi mô phỏng**:
   - `currency = 'VND'` (100%): Toàn bộ mô phỏng chỉ dùng đồng Việt Nam.
   - `country = 'VN'` (100%): Không có giao dịch quốc tế.
   - `created_at = '2026-08-06T03:00:00+00:00'` (100%): Metadata thời điểm sinh dữ liệu, không mang ý nghĩa hành vi.
2. **Nhóm thiếu hụt do đặc thù kênh thanh toán (Transfer vs Merchant)**:
   - `merchant_id` và `merchant_category_code` (100% NULL): Dữ liệu hiện tại chỉ mô phỏng chuyển khoản cá nhân/doanh nghiệp (`transfer`), chưa có giao dịch quẹt thẻ/thanh toán POS/E-commerce với Merchant.
3. **Nhóm trạng thái hệ thống (Execution Status)**:
   - `status = 'success'` (100%) và `failure_reason` (100% NULL): Toàn bộ giao dịch đều được ghi nhận thành công ở tầng xử lý kỹ thuật. Do đó, `status` không có sức phân biệt gian lận trong snapshot này.
   - `proxy_flag = 'false'` (100%): Trên bảng `transactions`, cờ proxy không được kích hoạt độc lập (tín hiệu proxy nằm trong bảng `login_sessions`).

**Hành động (Action Item)**: Đưa 8 cột này vào danh sách **`DROP_COLUMNS`** khi xây dựng Feature Matrix tại Notebook `03_feature_engineering` để giảm chiều dữ liệu và tối ưu chi phí tính toán.


---
<a id="3"></a>
## 3. Kiểm tra Tính duy nhất & Toàn vẹn Khóa (Uniqueness & Referential Integrity)

### 3.1 Tính Duy nhất của Khóa chính (Primary Key Uniqueness)

- **Mục tiêu**: Đảm bảo cột khóa chính (PK) của tất cả 10 bảng nghiệp vụ không chứa bất kỳ giá trị NULL nào và 100% giá trị là duy nhất, không bị trùng lặp khi hợp nhất 5 simulation runs độc lập.
- **Input**: Cột PK tương ứng của từng bảng (`transactions.transaction_id`, `customers.customer_id`, `accounts.account_id`, v.v.).
- **Output kỳ vọng**: Số lượng duplicates = 0 và nulls = 0 đối với tất cả các bảng (`PASS`).


In [50]:
PK_MAP = {
    "customers": "customer_id",
    "accounts": "account_id",
    "devices": "device_id",
    "login_sessions": "session_id",
    "beneficiaries": "beneficiary_id",
    "account_change_events": "change_event_id",
    "transactions": "transaction_id",
    "transaction_features": "transaction_id",
    "auth_events": "auth_event_id",
    "fraud_ground_truth": "fraud_event_id"
}

pk_results = []
for tbl, pk_col in PK_MAP.items():
    df = tables[tbl]
    total_rows = len(df)
    unique_pks = df[pk_col].nunique()
    null_pks = df[pk_col].isnull().sum()
    duplicate_count = total_rows - unique_pks
    
    status = "PASS" if (duplicate_count == 0 and null_pks == 0) else "FAIL"
    
    pk_results.append({
        "Bảng": tbl,
        "Cột PK": pk_col,
        "Tổng số dòng": f"{total_rows:,}",
        "Số PK duy nhất": f"{unique_pks:,}",
        "Số lượng Null": null_pks,
        "Số lượng Trùng lặp": duplicate_count,
        "Trạng thái": status
    })
    
    log_dq(
        check_name="PK Uniqueness",
        table_col=f"{tbl}.{pk_col}",
        actual=f"Duplicates: {duplicate_count}, Nulls: {null_pks}",
        expected="Duplicates: 0, Nulls: 0",
        status=status,
        action="Không cần hành động" if status == "PASS" else f"Cần deduplicate bảng {tbl} trước khi join"
    )

pk_df = pd.DataFrame(pk_results)
display(pk_df)


,Bảng,Cột PK,Tổng số dòng,Số PK duy nhất,Số lượng Null,Số lượng Trùng lặp,Trạng thái
0,customers,customer_id,"7,500","7,500",0,0,PASS
1,accounts,account_id,"9,651","9,651",0,0,PASS
2,devices,device_id,"10,625","10,625",0,0,PASS
3,login_sessions,session_id,"26,676","26,676",0,0,PASS
4,beneficiaries,beneficiary_id,"20,467","20,467",0,0,PASS
5,account_change_events,change_event_id,"2,573","2,573",0,0,PASS
6,transactions,transaction_id,"111,064","111,064",0,0,PASS
7,transaction_features,transaction_id,"111,064","111,064",0,0,PASS
8,auth_events,auth_event_id,"39,693","39,693",0,0,PASS
9,fraud_ground_truth,fraud_event_id,"1,900","1,900",0,0,PASS


### 3.2 Tính Toàn vẹn Khóa ngoại & Phát hiện Bản ghi Mồ côi (Foreign Key Integrity)

- **Mục tiêu**: Kiểm tra tính toàn vẹn tham chiếu (Referential Integrity): mọi khóa ngoại từ bảng con (VD: `transactions.account_id`, `transactions.customer_id`, `transactions.session_id`) đều phải tồn tại trong bảng cha tương ứng (`accounts`, `customers`, `login_sessions`), không để phát sinh bản ghi mồ côi (Orphan records).
- **Input**: 12 cặp quan hệ (Child Table.FK -> Parent Table.PK).
- **Output kỳ vọng**: 0 bản ghi mồ côi (`Orphan = 0`) cho các trường bắt buộc; riêng trường tùy chọn (`beneficiary_id` khi giao dịch là CREDIT) cho phép NULL nhưng nếu có giá trị thì phải map được vào bảng cha.


In [51]:
FK_CHECKS = [
    # Child table, FK col, Parent table, PK col, Optional (cho phép null)
    ("accounts", "customer_id", "customers", "customer_id", False),
    ("login_sessions", "account_id", "accounts", "account_id", False),
    ("login_sessions", "customer_id", "customers", "customer_id", False),
    ("login_sessions", "device_id", "devices", "device_id", False),
    ("beneficiaries", "account_id", "accounts", "account_id", False),
    ("account_change_events", "account_id", "accounts", "account_id", False),
    ("transactions", "account_id", "accounts", "account_id", False),
    ("transactions", "customer_id", "customers", "customer_id", False),
    ("transactions", "session_id", "login_sessions", "session_id", False),
    ("transactions", "device_id", "devices", "device_id", False),
    ("transactions", "beneficiary_id", "beneficiaries", "beneficiary_id", True), # Cho phép rỗng với CREDIT/cash
    ("transaction_features", "transaction_id", "transactions", "transaction_id", False)
]

fk_results = []
for child_tbl, fk_col, parent_tbl, pk_col, is_optional in FK_CHECKS:
    child_df = tables[child_tbl]
    parent_df = tables[parent_tbl]
    
    parent_keys = set(parent_df[pk_col].dropna())
    
    # Lọc các giá trị không null và không rỗng ở bảng con
    child_fks = child_df[fk_col].dropna()
    child_fks_valid = child_fks[child_fks != ""]
    
    orphan_mask = ~child_fks_valid.isin(parent_keys)
    orphan_count = orphan_mask.sum()
    
    status = "PASS" if orphan_count == 0 else "FAIL"
    
    fk_results.append({
        "Mối quan hệ (FK -> PK)": f"{child_tbl}.{fk_col} -> {parent_tbl}.{pk_col}",
        "Tổng số FK hợp lệ": f"{len(child_fks_valid):,}",
        "Bản ghi mồ côi (Orphans)": orphan_count,
        "Cho phép Null?": "Có" if is_optional else "Không",
        "Trạng thái": status
    })
    
    log_dq(
        check_name="FK Referential Integrity",
        table_col=f"{child_tbl}.{fk_col}",
        actual=f"{orphan_count} orphans",
        expected="0 orphans",
        status=status,
        action="Toàn vẹn quan hệ" if status == "PASS" else "Cần loại bỏ hoặc impute FK lỗi trước khi join"
    )

fk_df = pd.DataFrame(fk_results)
display(fk_df)


,Mối quan hệ (FK -> PK),Tổng số FK hợp lệ,Bản ghi mồ côi (Orphans),Cho phép Null?,Trạng thái
0,accounts.customer_id -> customers.customer_id,"9,651",0,Không,PASS
1,login_sessions.account_id -> accounts.account_id,"26,676",0,Không,PASS
2,login_sessions.customer_id -> customers.custom...,"26,676",0,Không,PASS
3,login_sessions.device_id -> devices.device_id,"26,676",0,Không,PASS
4,beneficiaries.account_id -> accounts.account_id,"20,467",0,Không,PASS
5,account_change_events.account_id -> accounts.a...,"2,573",0,Không,PASS
6,transactions.account_id -> accounts.account_id,"111,064",0,Không,PASS
7,transactions.customer_id -> customers.customer_id,"111,064",0,Không,PASS
8,transactions.session_id -> login_sessions.sess...,"111,064",0,Không,PASS
9,transactions.device_id -> devices.device_id,"111,064",0,Không,PASS


---
<a id="4"></a>
## 4. Kiểm tra Tính hợp lệ & Miền giá trị (Validity & Value Range Checks)

### 4.1 Kiểm định Ràng buộc Số học và Miền Giá trị Tiền tệ (Financial Range Constraints)

- **Mục tiêu**: Đảm bảo các chỉ số tài chính tuân thủ logic nghiệp vụ ngân hàng:
  1. Số tiền giao dịch phải dương (`amount > 0`).
  2. Số dư tài khoản không được âm (`balance_before >= 0` và `balance_after >= 0`).
  3. Hạn mức giao dịch và hạn mức ngày phải dương (`single_txn_limit > 0`, `daily_transfer_limit > 0`).
  4. Điểm rủi ro (Risk scores) của phiên, thiết bị và xác thực phải nằm trong khoảng chuẩn `[0, 100]`.
- **Input**: DataFrame `transactions`, `accounts`, `devices`, `login_sessions`, `auth_events`.
- **Output kỳ vọng**: 0 bản ghi vi phạm miền giá trị (`Violations = 0`, `PASS`).


In [52]:
range_checks = []

# 1. Kiểm tra transactions: amount > 0, balance >= 0
txn_df = tables["transactions"]
amount_num = pd.to_numeric(txn_df["amount"], errors="coerce")
bal_before_num = pd.to_numeric(txn_df["balance_before"], errors="coerce")
bal_after_num = pd.to_numeric(txn_df["balance_after"], errors="coerce")

v_amt = (amount_num <= 0).sum()
v_bal_b = (bal_before_num < 0).sum()
v_bal_a = (bal_after_num < 0).sum()

range_checks.extend([
    {"Trường dữ liệu": "transactions.amount", "Ràng buộc": "amount > 0", "Số lượng vi phạm": v_amt, "Status": "PASS" if v_amt == 0 else "FAIL"},
    {"Trường dữ liệu": "transactions.balance_before", "Ràng buộc": "balance_before >= 0", "Số lượng vi phạm": v_bal_b, "Status": "PASS" if v_bal_b == 0 else "FAIL"},
    {"Trường dữ liệu": "transactions.balance_after", "Ràng buộc": "balance_after >= 0", "Số lượng vi phạm": v_bal_a, "Status": "PASS" if v_bal_a == 0 else "FAIL"}
])

# 2. Kiểm tra accounts: single_txn_limit > 0, daily_transfer_limit > 0
acc_df = tables["accounts"]
single_limit = pd.to_numeric(acc_df["single_txn_limit"], errors="coerce")
daily_limit = pd.to_numeric(acc_df["daily_transfer_limit"], errors="coerce")

v_single = (single_limit <= 0).sum()
v_daily = (daily_limit <= 0).sum()
v_limit_order = (single_limit > daily_limit).sum() # Hạn mức 1 lần không thể vượt hạn mức ngày

range_checks.extend([
    {"Trường dữ liệu": "accounts.single_txn_limit", "Ràng buộc": "single_txn_limit > 0", "Số lượng vi phạm": v_single, "Status": "PASS" if v_single == 0 else "FAIL"},
    {"Trường dữ liệu": "accounts.daily_transfer_limit", "Ràng buộc": "daily_transfer_limit > 0", "Số lượng vi phạm": v_daily, "Status": "PASS" if v_daily == 0 else "FAIL"},
    {"Trường dữ liệu": "accounts.limit_consistency", "Ràng buộc": "single_limit <= daily_limit", "Số lượng vi phạm": v_limit_order, "Status": "PASS" if v_limit_order == 0 else "FAIL"}
])

# 3. Kiểm tra Risk Scores: nằm trong [0, 100]
dev_risk = pd.to_numeric(tables["devices"]["device_risk_score"], errors="coerce")
ses_risk = pd.to_numeric(tables["login_sessions"]["session_risk_score"], errors="coerce")
auth_risk = pd.to_numeric(tables["auth_events"]["auth_risk_score"], errors="coerce")

v_dev_r = ((dev_risk < 0) | (dev_risk > 100)).sum()
v_ses_r = ((ses_risk < 0) | (ses_risk > 100)).sum()
v_auth_r = ((auth_risk < 0) | (auth_risk > 100)).sum()

range_checks.extend([
    {"Trường dữ liệu": "devices.device_risk_score", "Ràng buộc": "0 <= score <= 100", "Số lượng vi phạm": v_dev_r, "Status": "PASS" if v_dev_r == 0 else "FAIL"},
    {"Trường dữ liệu": "login_sessions.session_risk_score", "Ràng buộc": "0 <= score <= 100", "Số lượng vi phạm": v_ses_r, "Status": "PASS" if v_ses_r == 0 else "FAIL"},
    {"Trường dữ liệu": "auth_events.auth_risk_score", "Ràng buộc": "0 <= score <= 100", "Số lượng vi phạm": v_auth_r, "Status": "PASS" if v_auth_r == 0 else "FAIL"}
])

range_df = pd.DataFrame(range_checks)
display(range_df)

for r in range_checks:
    log_dq(
        check_name="Range & Domain Constraint",
        table_col=r["Trường dữ liệu"],
        actual=f"{r['Số lượng vi phạm']} vi phạm",
        expected=r["Ràng buộc"],
        status=r["Status"],
        action="Dữ liệu tài chính hợp lệ" if r["Status"] == "PASS" else "Cần clip/capping hoặc loại trừ bản ghi vi phạm"
    )


,Trường dữ liệu,Ràng buộc,Số lượng vi phạm,Status
0,transactions.amount,amount > 0,0,PASS
1,transactions.balance_before,balance_before >= 0,0,PASS
2,transactions.balance_after,balance_after >= 0,0,PASS
3,accounts.single_txn_limit,single_txn_limit > 0,0,PASS
4,accounts.daily_transfer_limit,daily_transfer_limit > 0,0,PASS
5,accounts.limit_consistency,single_limit <= daily_limit,0,PASS
6,devices.device_risk_score,0 <= score <= 100,0,PASS
7,login_sessions.session_risk_score,0 <= score <= 100,0,PASS
8,auth_events.auth_risk_score,0 <= score <= 100,0,PASS


### 4.2 Kiểm định Danh mục Giá trị Cho phép (Categorical Whitelist Validation)

- **Mục tiêu**: Đảm bảo tất cả các trường phân loại (Categorical columns) chỉ nhận các giá trị nằm trong danh mục nghiệp vụ hợp lệ (Domain Whitelist), không xuất hiện giá trị lạ, lỗi chính tả hay định dạng ngoài chuẩn.
- **Input**: Các cột phân loại trên `transactions`, `accounts`, `customers`, `beneficiaries`.
- **Output kỳ vọng**: 100% giá trị thuộc whitelist (`Unexpected categories = 0`, `PASS`).


In [53]:
WHITELISTS = {
    ("transactions", "direction"): ["DEBIT", "CREDIT"],
    # Đã cập nhật đủ 4 loại giao dịch nghiệp vụ thực tế
    ("transactions", "transaction_type"): ["transfer", "cash_withdrawal", "bill_payment", "merchant_settlement"],
    ("transactions", "channel"): ["mobile", "web", "api", "branch", "atm"],
    ("accounts", "account_type"): ["CASA", "payroll", "savings", "business"],
    ("accounts", "status"): ["active", "dormant", "frozen", "closed"],
    ("customers", "customer_type"): ["individual", "sme"],
    ("customers", "gender"): ["M", "F", "U"],
    ("customers", "customer_status"): ["active", "inactive", "blocked", "closed"],
    ("customers", "kyc_level"): ["basic", "standard", "biometric_verified", "enhanced"],
    ("customers", "base_risk_level"): ["Low", "Medium", "High"],
    ("beneficiaries", "beneficiary_risk_level"): ["Low", "Medium", "High"],
    ("beneficiaries", "status"): ["active", "removed"]
}

cat_results = []
for (tbl, col), allowed in WHITELISTS.items():
    actual_vals = set(tables[tbl][col].dropna().unique())
    unexpected = actual_vals - set(allowed)
    status = "PASS" if len(unexpected) == 0 else "FAIL"
    
    cat_results.append({
        "Bảng.Cột": f"{tbl}.{col}",
        "Whitelist cho phép": str(allowed),
        "Giá trị thực tế xuất hiện": str(sorted(list(actual_vals))),
        "Giá trị ngoài danh mục": str(list(unexpected)) if unexpected else "Không có",
        "Trạng thái": status
    })
    
    log_dq(
        check_name="Categorical Whitelist",
        table_col=f"{tbl}.{col}",
        actual=f"Unexpected: {len(unexpected)}",
        expected=f"Chỉ nằm trong {allowed}",
        status=status,
        action="Chuẩn hóa category" if status == "PASS" else f"Cần map lại các giá trị: {unexpected}"
    )

cat_df = pd.DataFrame(cat_results)
display(cat_df)


,Bảng.Cột,Whitelist cho phép,Giá trị thực tế xuất hiện,Giá trị ngoài danh mục,Trạng thái
0,transactions.direction,"['DEBIT', 'CREDIT']","['CREDIT', 'DEBIT']",Không có,PASS
1,transactions.transaction_type,"['transfer', 'cash_withdrawal', 'bill_payment'...","['bill_payment', 'cash_withdrawal', 'merchant_...",Không có,PASS
2,transactions.channel,"['mobile', 'web', 'api', 'branch', 'atm']","['api', 'atm', 'branch', 'mobile', 'web']",Không có,PASS
3,accounts.account_type,"['CASA', 'payroll', 'savings', 'business']","['CASA', 'business', 'payroll', 'savings']",Không có,PASS
4,accounts.status,"['active', 'dormant', 'frozen', 'closed']","['active', 'closed', 'dormant', 'frozen']",Không có,PASS
5,customers.customer_type,"['individual', 'sme']","['individual', 'sme']",Không có,PASS
6,customers.gender,"['M', 'F', 'U']","['F', 'M', 'U']",Không có,PASS
7,customers.customer_status,"['active', 'inactive', 'blocked', 'closed']","['active', 'blocked', 'closed', 'inactive']",Không có,PASS
8,customers.kyc_level,"['basic', 'standard', 'biometric_verified', 'e...","['basic', 'biometric_verified', 'enhanced', 's...",Không có,PASS
9,customers.base_risk_level,"['Low', 'Medium', 'High']","['High', 'Low', 'Medium']",Không có,PASS


### 4.3 Kiểm tra Tính Toàn vẹn của Định dạng Hash Bảo mật (SHA-256 Format Integrity)

- **Mục tiêu**: Trong hệ thống ngân hàng, các dữ liệu PII (số CMND/CCCD, số điện thoại, email, số tài khoản thụ hưởng) đều được hash ẩn danh bằng thuật toán SHA-256. Cần kiểm tra để đảm bảo toàn bộ các chuỗi hash này:
  1. Có độ dài chuẩn xác bằng đúng 64 ký tự Hex (`length == 64`).
  2. Chỉ chứa các ký tự hợp lệ (`0-9`, `a-f`).
- **Input**: Các cột hash trên `customers`, `accounts`, `transactions`, `beneficiaries`.
- **Output kỳ vọng**: 100% hash hợp lệ chuẩn SHA-256 (`Invalid Hash = 0`, `PASS`).


In [54]:
import re

HASH_COLS = [
    ("customers", "id_number_hash"),
    ("customers", "phone_hash"),
    ("customers", "email_hash"),
    ("accounts", "account_no_hash"),
    ("beneficiaries", "beneficiary_account_hash"),
    ("transactions", "counterparty_account_hash")
]

hash_regex = re.compile(r"^[0-9a-f]{64}$")
hash_results = []

for tbl, col in HASH_COLS:
    series = tables[tbl][col].dropna()
    series_valid = series[series != ""]
    
    # Kiểm tra độ dài và định dạng regex
    invalid_mask = ~series_valid.str.match(hash_regex)
    invalid_count = invalid_mask.sum()
    sample_hash = series_valid.iloc[0] if len(series_valid) > 0 else "N/A"
    
    status = "PASS" if invalid_count == 0 else "FAIL"
    
    hash_results.append({
        "Bảng.Cột Hash": f"{tbl}.{col}",
        "Tổng số bản ghi kiểm tra": f"{len(series_valid):,}",
        "Số hash không chuẩn SHA-256": invalid_count,
        "Hash mẫu": f"{sample_hash[:16]}...{sample_hash[-8:]}",
        "Trạng thái": status
    })
    
    log_dq(
        check_name="SHA-256 Hash Format",
        table_col=f"{tbl}.{col}",
        actual=f"{invalid_count} non-SHA256 hashes",
        expected="64-char Hex (SHA-256)",
        status=status,
        action="Định dạng bảo mật đạt chuẩn" if status == "PASS" else "Lỗi mã hóa PII hash"
    )

hash_df = pd.DataFrame(hash_results)
display(hash_df)


,Bảng.Cột Hash,Tổng số bản ghi kiểm tra,Số hash không chuẩn SHA-256,Hash mẫu,Trạng thái
0,customers.id_number_hash,"7,500",0,74f3e37238eeeb6e...42ddd8d9,PASS
1,customers.phone_hash,"7,500",0,934e04d1c54a1008...a2ce78d4,PASS
2,customers.email_hash,"7,500",0,bcc54c4b57677b4c...e6598fcf,PASS
3,accounts.account_no_hash,"9,651",0,56c0720ce77afde2...41194a6a,PASS
4,beneficiaries.beneficiary_account_hash,"20,467",0,9550839c376209b1...f48d9826,PASS
5,transactions.counterparty_account_hash,"111,064",0,7022c5841ace8ba6...ab1d6ad8,PASS


---
<a id="5"></a>
## 5. Phân tích & Phân loại Giá trị Thiếu (Completeness & Missing Value Taxonomy)

Trong bài toán Fraud Detection thời gian thực, **missing values không đơn thuần là dữ liệu bị mất (data loss) để fill bừa bằng 0 hay median**. 

Cần phân loại giá trị thiếu thành 4 nhóm bản chất nghiệp vụ:
1. **Not Applicable (Không áp dụng)**: Thuộc tính không thể tồn tại với loại sự kiện đó (VD: giao dịch tiền vào `CREDIT` thì không có `beneficiary_id`).
2. **No Prior History (Chưa có lịch sử phát sinh)**: Khách hàng chưa từng thực hiện hành động này trước đây (VD: chưa từng đổi thông tin bảo mật nên `time_since_sensitive_change` bị trống).
3. **Absent Risk Relation (Không thuộc mạng lưới rủi ro)**: Khách hàng/người nhận không nằm trong danh sách đen/cụm tài khoản mule (`mule_cluster_id` rỗng).
4. **Data Failure / System Missing (Lỗi thu thập dữ liệu)**: Dữ liệu đáng lẽ phải có nhưng bị mất do lỗi hệ thống.

Mỗi nhóm đòi hỏi chiến lược xử lý (Imputation / Missing Indicator) hoàn toàn khác nhau.


### 5.1 Khảo sát Ma trận Khuyết thiếu trên Toàn bộ các Bảng (Missing Matrix Profiling)

- **Mục tiêu**: Quét toàn bộ các cột trên 11 bảng để tính toán tỷ lệ khuyết thiếu thực tế (bao gồm cả giá trị `NULL`, `NaN` và chuỗi rỗng `""`).
- **Input**: DataFrame của tất cả 11 bảng.
- **Output kỳ vọng**: Bảng thống kê các cột có phát sinh missing, tỷ lệ missing (%) và phân loại sơ bộ.


In [55]:
missing_records = []

for tbl_name, df in tables.items():
    total_len = len(df)
    if total_len == 0:
        continue
    for col in df.columns:
        # Đếm cả NaN/None và chuỗi rỗng ""
        null_count = df[col].isnull().sum() + (df[col] == "").sum()
        if null_count > 0:
            missing_rate = null_count / total_len
            missing_records.append({
                "Bảng": tbl_name,
                "Cột": col,
                "Số lượng thiếu": null_count,
                "Tổng số dòng": total_len,
                "Tỷ lệ thiếu (%)": missing_rate * 100
            })

missing_df = pd.DataFrame(missing_records).sort_values(by="Tỷ lệ thiếu (%)", ascending=False)
print(f"=== PHÁT HIỆN {len(missing_df)} CỘT CÓ GIÁ TRỊ THIẾU TRÊN TOÀN BỘ DATASET ===")
display(missing_df)


=== PHÁT HIỆN 17 CỘT CÓ GIÁ TRỊ THIẾU TRÊN TOÀN BỘ DATASET ===


,Bảng,Cột,Số lượng thiếu,Tổng số dòng,Tỷ lệ thiếu (%)
2,transactions,merchant_id,111064,111064,100.0000
4,transactions,failure_reason,111064,111064,100.0000
3,transactions,merchant_category_code,111064,111064,100.0000
16,fraud_ground_truth,primary_application_id,1900,1900,100.0000
14,auth_events,change_event_id,39618,39693,99.8110
11,beneficiaries,mule_cluster_id,19862,20467,97.0440
10,login_sessions,failure_reason,25876,26676,97.0010
9,accounts,dormant_since,9205,9651,95.3787
13,auth_events,session_id,37629,39693,94.8001
7,transaction_features,time_since_sensitive_change_minutes,102191,111064,92.0109


#### Nhận xét về Giá trị Thiếu (Missing Values Insights)

Toàn bộ dataset phát sinh **17 cột có giá trị thiếu trên 8 bảng**, 3 bảng còn lại (`customers`, `devices`, `account_change_events`) **đạt 100% completeness**:

1. **Khớp nối nghiệp vụ hoàn hảo**:
   - `transactions.beneficiary_id` thiếu **18.35%** (do là giao dịch tiền vào `CREDIT` hoặc rút tiền mặt).
   - `transaction_features.time_since_beneficiary_added_minutes` cũng thiếu **đúng 18.35%** tương ứng.
2. **Bản chất missing hợp lệ**:
   - `auth_events`: 3 cột context (`transaction_id`, `session_id`, `change_event_id`) loại trừ lẫn nhau (chỉ 1 cột có giá trị/dòng).
   - `time_since_sensitive_change_minutes` (thiếu 93.14%): Do phần lớn tài khoản an toàn chưa từng đổi mật khẩu/SĐT.
   - `scenario_event_entities.sample_weight` (thiếu 2.44% = đúng 100 dòng): Thuộc về scenario `TXN-03` (tấn công brute-force ở cấp session/account, không sinh transaction).

**Kết luận**: Không có cột nào bị thiếu do lỗi hệ thống (Data Loss). Tất cả missing đều mang ý nghĩa nghiệp vụ rõ ràng và sẽ được xử lý theo bảng chiến lược ở phần 5.2.


### 5.2 Phân loại Bản chất Nghiệp vụ & Đề xuất Chiến lược Xử lý (Imputation Policy)

- **Mục tiêu**: Phân tích chi tiết nguyên nhân khuyết thiếu của các cột trọng yếu và thiết lập bảng **Quy tắc Xử lý Missing (Missing Handling Strategy)** chuẩn bị cho Notebook `03_feature_engineering`.
- **Input**: DataFrame `transactions`, `transaction_features`, `accounts`, `beneficiaries`.
- **Output kỳ vọng**: Bảng quy tắc xử lý (Imputation Strategy, Indicator Flag, SAS Live Compatibility) cho từng cột bị thiếu.


In [56]:
# 1. Kiểm tra tính hợp lý của missing trên transactions.beneficiary_id
txn_df = tables["transactions"]
no_bene = txn_df[txn_df["beneficiary_id"].isna() | (txn_df["beneficiary_id"] == "")]
direction_dist = no_bene["direction"].value_counts()
type_dist = no_bene["transaction_type"].value_counts()

print("Phân phối loại giao dịch khi beneficiary_id bị trống:")
print("Direction:", direction_dist.to_dict())
print("Type:", type_dist.to_dict())

# 2. Xây dựng Bảng Phân loại & Chiến lược Xử lý Missing
taxonomy_data = [
    {
        "Bảng.Cột": "transactions.beneficiary_id",
        "Tỷ lệ thiếu": f"{(txn_df['beneficiary_id'].isna() | (txn_df['beneficiary_id'] == '')).mean():.2%}",
        "Nhóm Bản chất": "1. Not Applicable",
        "Nguyên nhân Nghiệp vụ": "Giao dịch CREDIT (tiền vào) hoặc Rút tiền mặt ATM không có người thụ hưởng",
        "Chiến lược Xử lý (Feature Engineering)": "Giữ nguyên rỗng; Tạo cờ `has_beneficiary = 0/1`"
    },
    {
        "Bảng.Cột": "transactions.counterparty_internal_account_id",
        "Tỷ lệ thiếu": f"{(txn_df['counterparty_internal_account_id'].isna() | (txn_df['counterparty_internal_account_id'] == '')).mean():.2%}",
        "Nhóm Bản chất": "3. Absent Risk Relation",
        "Nguyên nhân Nghiệp vụ": "Phần lớn giao dịch chuyển khoản ra ngoài ngân hàng (External Bank)",
        "Chiến lược Xử lý (Feature Engineering)": "Impute 'EXTERNAL'; Tạo feature `is_internal_transfer = 0/1`"
    },
    {
        "Bảng.Cột": "transaction_features.time_since_sensitive_change_minutes",
        "Tỷ lệ thiếu": f"{(tables['transaction_features']['time_since_sensitive_change_minutes'].isna() | (tables['transaction_features']['time_since_sensitive_change_minutes'] == '')).mean():.2%}",
        "Nhóm Bản chất": "2. No Prior History",
        "Nguyên nhân Nghiệp vụ": "Khách hàng chưa từng có sự kiện đổi pass/phone/device trong lịch sử",
        "Chiến lược Xử lý (Feature Engineering)": "Impute giá trị cực lớn (VD: 999999 min); Tạo cờ `has_prior_sensitive_change = 0/1`"
    },
    {
        "Bảng.Cột": "transaction_features.amount_to_median_ratio",
        "Tỷ lệ thiếu": f"{(tables['transaction_features']['amount_to_median_ratio'].isna() | (tables['transaction_features']['amount_to_median_ratio'] == '')).mean():.2%}",
        "Nhóm Bản chất": "2. No Prior History",
        "Nguyên nhân Nghiệp vụ": "Giao dịch đầu tiên của tài khoản (chưa có lịch sử để tính median amount)",
        "Chiến lược Xử lý (Feature Engineering)": "Impute 1.0 (tương đương amount = baseline); Tạo cờ `is_first_transaction = 0/1`"
    },
    {
        "Bảng.Cột": "accounts.dormant_since",
        "Tỷ lệ thiếu": f"{(tables['accounts']['dormant_since'].isna() | (tables['accounts']['dormant_since'] == '')).mean():.2%}",
        "Nhóm Bản chất": "1. Not Applicable",
        "Nguyên nhân Nghiệp vụ": "Tài khoản đang active bình thường, không bị đóng băng/dormant",
        "Chiến lược Xử lý (Feature Engineering)": "Không impute timestamp; Sử dụng cột trạng thái `accounts.status == 'dormant'`"
    },
    {
        "Bảng.Cột": "beneficiaries.mule_cluster_id",
        "Tỷ lệ thiếu": f"{(tables['beneficiaries']['mule_cluster_id'].isna() | (tables['beneficiaries']['mule_cluster_id'] == '')).mean():.2%}",
        "Nhóm Bản chất": "3. Absent Risk Relation",
        "Nguyên nhân Nghiệp vụ": "Người thụ hưởng bình thường, không nằm trong mạng lưới mule nghi vấn",
        "Chiến lược Xử lý (Feature Engineering)": "Impute 'NONE'; Tạo cờ `is_mule_cluster_linked = 0/1`"
    }
]

taxonomy_df = pd.DataFrame(taxonomy_data)
display(taxonomy_df)

for r in taxonomy_data:
    log_dq(
        check_name="Missing Taxonomy Audit",
        table_col=r["Bảng.Cột"],
        actual=f"Thiếu {r['Tỷ lệ thiếu']}",
        expected="Phân loại & có chiến lược xử lý",
        status="PASS",
        action=r["Chiến lược Xử lý (Feature Engineering)"]
    )


Phân phối loại giao dịch khi beneficiary_id bị trống:
Direction: {'CREDIT': 20107, 'DEBIT': 275}
Type: {'transfer': 19768, 'merchant_settlement': 339, 'cash_withdrawal': 275}


,Bảng.Cột,Tỷ lệ thiếu,Nhóm Bản chất,Nguyên nhân Nghiệp vụ,Chiến lược Xử lý (Feature Engineering)
0,transactions.beneficiary_id,18.35%,1. Not Applicable,Giao dịch CREDIT (tiền vào) hoặc Rút tiền mặt ...,Giữ nguyên rỗng; Tạo cờ `has_beneficiary = 0/1`
1,transactions.counterparty_internal_account_id,82.84%,3. Absent Risk Relation,Phần lớn giao dịch chuyển khoản ra ngoài ngân ...,Impute 'EXTERNAL'; Tạo feature `is_internal_tr...
2,transaction_features.time_since_sensitive_chan...,92.01%,2. No Prior History,Khách hàng chưa từng có sự kiện đổi pass/phone...,Impute giá trị cực lớn (VD: 999999 min); Tạo c...
3,transaction_features.amount_to_median_ratio,8.69%,2. No Prior History,Giao dịch đầu tiên của tài khoản (chưa có lịch...,Impute 1.0 (tương đương amount = baseline); Tạ...
4,accounts.dormant_since,95.38%,1. Not Applicable,"Tài khoản đang active bình thường, không bị đó...",Không impute timestamp; Sử dụng cột trạng thái...
5,beneficiaries.mule_cluster_id,97.04%,3. Absent Risk Relation,"Người thụ hưởng bình thường, không nằm trong m...",Impute 'NONE'; Tạo cờ `is_mule_cluster_linked ...


---
<a id="6"></a>
## 6. Kiểm tra Tính nhất quán & Chuỗi Nghiệp vụ (Consistency & Business Invariants)

### 6.1 Tính Liên tục của Chuỗi Số dư Tài khoản (Balance Chain Continuity)

- **Mục tiêu**: Trong hệ thống Core Banking, số dư tài khoản phải bảo toàn tính liên tục theo chuỗi thời gian:
  $$\text{Số dư trước GD}(T_{i+1}) = \text{Số dư sau GD}(T_i)$$
  Nếu chuỗi số dư bị đứt đoạn giữa các giao dịch liên tiếp, feature tính toán rolling balance hoặc tỷ lệ rút tiền sẽ bị sai lệch hoàn toàn.
- **Input**: DataFrame `transactions`, sắp xếp theo `account_id` và `transaction_at`.
- **Output kỳ vọng**: Số lượng cặp giao dịch bị lệch số dư = 0 (`Broken chains = 0`, `PASS`).


In [57]:
txn_df = tables["transactions"].copy()
txn_df["amount_num"] = pd.to_numeric(txn_df["amount"], errors="coerce")
txn_df["bal_before_num"] = pd.to_numeric(txn_df["balance_before"], errors="coerce")
txn_df["bal_after_num"] = pd.to_numeric(txn_df["balance_after"], errors="coerce")
txn_df["txn_time"] = pd.to_datetime(txn_df["transaction_at"], utc=True)

# Sắp xếp giao dịch theo account và thời gian
txn_sorted = txn_df.sort_values(by=["account_id", "txn_time"]).reset_index(drop=True)

# 1. Kiểm tra tính toán nội tại của từng giao dịch (Arithmetic Check):
# DEBIT: balance_after = balance_before - amount
# CREDIT: balance_after = balance_before + amount
debit_mask = txn_sorted["direction"] == "DEBIT"
credit_mask = txn_sorted["direction"] == "CREDIT"

expected_after = pd.Series(index=txn_sorted.index, dtype=float)
expected_after[debit_mask] = (txn_sorted.loc[debit_mask, "bal_before_num"] - txn_sorted.loc[debit_mask, "amount_num"]).clip(lower=0.0)
expected_after[credit_mask] = txn_sorted.loc[credit_mask, "bal_before_num"] + txn_sorted.loc[credit_mask, "amount_num"]

arithmetic_diff = (txn_sorted["bal_after_num"] - expected_after).abs()
arithmetic_errors = (arithmetic_diff > 0.05).sum() # Cho phép sai số làm tròn 0.05 VND

# 2. Kiểm tra tính liên tục giữa 2 giao dịch liên tiếp (Chain Continuity)
txn_sorted["prev_account"] = txn_sorted["account_id"].shift(1)
txn_sorted["prev_bal_after"] = txn_sorted["bal_after_num"].shift(1)

# Chỉ so sánh khi 2 dòng liên tiếp thuộc cùng 1 account
same_acc_mask = (txn_sorted["account_id"] == txn_sorted["prev_account"])
chain_diff = (txn_sorted.loc[same_acc_mask, "bal_before_num"] - txn_sorted.loc[same_acc_mask, "prev_bal_after"]).abs()
chain_breaks = (chain_diff > 0.05).sum()

print("=== KẾT QUẢ KIỂM TRA CHUỖI SỐ DƯ (BALANCE CONTINUITY) ===")
print(f"- Lỗi tính toán số học (Arithmetic errors trong 1 GD): {arithmetic_errors:,}")
print(f"- Số cặp giao dịch bị đứt đoạn chuỗi số dư (Broken chains giữa 2 GD): {chain_breaks:,}")

status_chain = "PASS" if (chain_breaks == 0 and arithmetic_errors == 0) else "FAIL"

log_dq(
    check_name="Balance Arithmetic Consistency",
    table_col="transactions.balance_after",
    actual=f"{arithmetic_errors} arithmetic errors",
    expected="0 arithmetic errors",
    status="PASS" if arithmetic_errors == 0 else "FAIL",
    action="Số học giao dịch chuẩn xác" if arithmetic_errors == 0 else "Lỗi tính toán balance_after"
)

log_dq(
    check_name="Balance Chain Continuity",
    table_col="transactions.balance_chain",
    actual=f"{chain_breaks} broken chains",
    expected="0 broken chains",
    status=status_chain,
    action="Chuỗi số dư liên tục hoàn hảo" if status_chain == "PASS" else "Cần chạy recompute_balances.py"
)


=== KẾT QUẢ KIỂM TRA CHUỖI SỐ DƯ (BALANCE CONTINUITY) ===
- Lỗi tính toán số học (Arithmetic errors trong 1 GD): 0
- Số cặp giao dịch bị đứt đoạn chuỗi số dư (Broken chains giữa 2 GD): 0


### 6.2 Tính Nhất quán Dữ liệu Liên bảng (Cross-Table Geographic & Entity Consistency)

- **Mục tiêu**: Kiểm tra tính nhất quán thông tin chéo giữa các bảng:
  1. Tỉnh/thành phố chi nhánh của tài khoản (`accounts.home_province`) phải khớp với tỉnh/thành phố thường trú của khách hàng (`customers.province`).
  2. Mỗi tài khoản trong `accounts` chỉ được liên kết với duy nhất 1 `customer_id`.
- **Input**: DataFrame `accounts`, `customers`.
- **Output kỳ vọng**: 100% khớp tính nhất quán liên bảng (`Inconsistencies = 0`, `PASS`).


In [58]:
acc_cust_df = tables["accounts"].merge(
    tables["customers"][["customer_id", "province"]],
    on="customer_id",
    how="left",
    suffixes=("_acc", "_cust")
)

# 1. So sánh tỉnh/thành phố
prov_mismatch = (acc_cust_df["home_province"] != acc_cust_df["province"]).sum()

# 2. Kiểm tra 1 account có bị gán cho nhiều customer_id không
acc_owner_count = tables["accounts"].groupby("account_id")["customer_id"].nunique()
multi_owner_accs = (acc_owner_count > 1).sum()

print("=== KẾT QUẢ KIỂM TRA NHẤT QUÁN LIÊN BẢNG ===")
print(f"- Lệch địa bàn cư trú vs chi nhánh (Home Province Mismatch): {prov_mismatch:,}")
print(f"- Tài khoản gán nhiều chủ sở hữu (Multi-owner accounts): {multi_owner_accs:,}")

log_dq(
    check_name="Cross-Table Geographic Consistency",
    table_col="accounts.home_province vs customers.province",
    actual=f"{prov_mismatch} mismatches",
    expected="0 mismatches (cùng địa bàn)",
    status="PASS" if prov_mismatch == 0 else "WARN",
    action="Nhất quán địa bàn" if prov_mismatch == 0 else "Ghi nhận nghiệp vụ mở tài khoản khác tỉnh"
)

log_dq(
    check_name="Account-Customer 1:1 Mapping",
    table_col="accounts.account_id -> customer_id",
    actual=f"{multi_owner_accs} multi-owner",
    expected="0 multi-owner",
    status="PASS" if multi_owner_accs == 0 else "FAIL",
    action="1 account thuộc duy nhất 1 customer" if multi_owner_accs == 0 else "Lỗi gán chủ tài khoản"
)


=== KẾT QUẢ KIỂM TRA NHẤT QUÁN LIÊN BẢNG ===
- Lệch địa bàn cư trú vs chi nhánh (Home Province Mismatch): 0
- Tài khoản gán nhiều chủ sở hữu (Multi-owner accounts): 0


---
<a id="7"></a>
## 7. Kiểm tra Tính nhân quả Thời gian (Timeliness & Causality Checks)

Trong phát hiện gian lận, **tính nhân quả thời gian (Causality)** là quy tắc bất khả xâm phạm:
1. **Phiên đăng nhập**: Giao dịch phải phát sinh *trong khoảng thời gian phiên hợp lệ* ($\text{login\_at} \le \text{transaction\_at} \le \text{session\_end\_at}$).
2. **Người thụ hưởng**: Người nhận tiền phải được thêm vào danh bạ *trước khi lệnh chuyển tiền xảy ra* ($\text{beneficiary.added\_at} \le \text{transaction\_at}$).
3. **Mở tài khoản**: Tài khoản phải được mở *trước hoặc cùng ngày với giao dịch đầu tiên* ($\text{open\_date} \le \text{transaction\_at}$).

Nếu vi phạm causality, các feature tính toán rolling/lookback sẽ bị âm (Look-ahead Leakage) hoặc vô nghĩa.


### 7.1 Kiểm định Dòng Thời gian giữa Giao dịch, Phiên Đăng nhập và Người Thụ hưởng

- **Mục tiêu**: Đo lường số lượng bản ghi vi phạm tính nhân quả giữa thời điểm giao dịch (`transaction_at`) với:
  - Thời điểm đăng nhập (`login_at`) và kết thúc phiên (`session_end_at`).
  - Thời điểm thêm người thụ hưởng (`added_at`).
  - Ngày mở tài khoản (`open_date`).
- **Input**: DataFrame `transactions`, `login_sessions`, `beneficiaries`, `accounts`.
- **Output kỳ vọng**: 0 bản ghi vi phạm trên tập kịch bản gian lận (Fraud/Hard-negative) và tỷ lệ vi phạm trên nền background < 2% (nếu có do giới hạn mô phỏng).


In [59]:
txn_time_df = tables["transactions"].copy()
ses_map = tables["login_sessions"].set_index("session_id")
ben_map = tables["beneficiaries"].set_index("beneficiary_id")
acc_map = tables["accounts"].set_index("account_id")

# Parse Datetime chuẩn UTC
txn_time_df["txn_at"] = pd.to_datetime(txn_time_df["transaction_at"], utc=True)
txn_time_df["ses_login_at"] = pd.to_datetime(txn_time_df["session_id"].map(ses_map["login_at"]), utc=True)
txn_time_df["ses_end_at"] = pd.to_datetime(txn_time_df["session_id"].map(ses_map["session_end_at"]), utc=True)
txn_time_df["ben_added_at"] = pd.to_datetime(txn_time_df["beneficiary_id"].map(ben_map["added_at"]), utc=True)
txn_time_df["acc_open_date"] = pd.to_datetime(txn_time_df["account_id"].map(acc_map["open_date"]), utc=True)

# 1. Kiểm tra Causality với Session
before_login = txn_time_df[txn_time_df["txn_at"] < txn_time_df["ses_login_at"]]
after_end = txn_time_df[txn_time_df["txn_at"] > txn_time_df["ses_end_at"]]

# 2. Kiểm tra Causality với Beneficiary
before_bene = txn_time_df[txn_time_df["ben_added_at"].notna() & (txn_time_df["txn_at"] < txn_time_df["ben_added_at"])]

# 3. Kiểm tra Causality với Ngày mở tài khoản
before_acc = txn_time_df[txn_time_df["txn_at"].dt.date < txn_time_df["acc_open_date"].dt.date]

causality_results = [
    {
        "Kiểm tra Nhân quả (Causality Check)": "Giao dịch trước khi Login (txn_at < login_at)",
        "Số dòng vi phạm": len(before_login),
        "Tỷ lệ (%)": f"{len(before_login) / len(txn_time_df):.4%}",
        "Trạng thái": "PASS" if len(before_login) == 0 else "FAIL"
    },
    {
        "Kiểm tra Nhân quả (Causality Check)": "Giao dịch sau khi Phiên kết thúc (txn_at > session_end_at)",
        "Số dòng vi phạm": len(after_end),
        "Tỷ lệ (%)": f"{len(after_end) / len(txn_time_df):.4%}",
        "Trạng thái": "PASS" if len(after_end) == 0 else "FAIL"
    },
    {
        "Kiểm tra Nhân quả (Causality Check)": "Giao dịch trước khi Thêm Beneficiary (txn_at < added_at)",
        "Số dòng vi phạm": len(before_bene),
        "Tỷ lệ (%)": f"{len(before_bene) / len(txn_time_df):.4%}",
        "Trạng thái": "PASS" if len(before_bene) == 0 else "FAIL"
    },
    {
        "Kiểm tra Nhân quả (Causality Check)": "Giao dịch trước Ngày mở tài khoản (txn_at < open_date)",
        "Số dòng vi phạm": len(before_acc),
        "Tỷ lệ (%)": f"{len(before_acc) / len(txn_time_df):.4%}",
        "Trạng thái": "PASS" if len(before_acc) == 0 else "WARN"
    }
]

causality_df = pd.DataFrame(causality_results)
display(causality_df)

for r in causality_results:
    log_dq(
        check_name="Timeline Causality",
        table_col=r["Kiểm tra Nhân quả (Causality Check)"],
        actual=f"{r['Số dòng vi phạm']} vi phạm ({r['Tỷ lệ (%)']})",
        expected="0 vi phạm",
        status=r["Trạng thái"],
        action="Nhân quả thời gian hoàn hảo" if r["Trạng thái"] == "PASS" else "Clip/fill hợp lệ khi tính feature tuổi tài khoản"
    )


,Kiểm tra Nhân quả (Causality Check),Số dòng vi phạm,Tỷ lệ (%),Trạng thái
0,Giao dịch trước khi Login (txn_at < login_at),0,0.0000%,PASS
1,Giao dịch sau khi Phiên kết thúc (txn_at > ses...,0,0.0000%,PASS
2,Giao dịch trước khi Thêm Beneficiary (txn_at <...,0,0.0000%,PASS
3,Giao dịch trước Ngày mở tài khoản (txn_at < op...,0,0.0000%,PASS


---
<a id="8"></a>
## 8. Kiểm tra Độ ổn định Đa nguồn (Cross-Run Distribution Stability)

### Khảo sát Phân phối Thống kê qua 5 Simulation Runs Độc lập

- **Mục tiêu**: Đảm bảo quá trình sinh dữ liệu từ 5 random seeds (`RUN_TXN_TRAIN_001` → `005`) có sự nhất quán cao về mặt phân phối:
  1. Số lượng khách hàng (`1.500/run`) và số lượng giao dịch (`~22.000/run`).
  2. Tỷ lệ gian lận (`fraud_rate_% ≈ 2.14% - 2.16%`).
  3. Tỷ lệ ca khó hợp lệ (`hard_neg_rate_% ≈ 1.39% - 1.51%`).
  4. Phân phối số tiền giao dịch (Mean & Median amount).
- **Input**: DataFrame `transactions` kết hợp với `scenario_event_entities`.
- **Output kỳ vọng**: Độ biến thiên giữa 5 runs là cực nhỏ ($CV < 5\%$), khẳng định dữ liệu không bị lệch ngẫu nhiên do seed (Seed-bias free).


In [60]:
# Gán nhãn tạm thời để đo lường độ ổn định phân phối giữa các run
bridge_df = tables["scenario_event_entities"]
bridge_txn = bridge_df[bridge_df["entity_type"] == "transaction"].set_index("entity_id")

txn_run_df = tables["transactions"].copy()
txn_run_df["target_fraud"] = txn_run_df["transaction_id"].map(bridge_txn["target_fraud"]).fillna("0").astype(int)
txn_run_df["hard_negative"] = txn_run_df["transaction_id"].map(bridge_txn["hard_negative"]).fillna("0").astype(int)
txn_run_df["amount_num"] = pd.to_numeric(txn_run_df["amount"], errors="coerce")

run_summary = txn_run_df.groupby("simulation_run_id").agg(
    Tong_Giao_Dich=("transaction_id", "count"),
    So_Khach_Hang=("customer_id", "nunique"),
    So_Tai_Khoan=("account_id", "nunique"),
    So_GD_Fraud=("target_fraud", "sum"),
    So_GD_Hard_Neg=("hard_negative", "sum"),
    So_Tien_Trung_Binh=("amount_num", "mean"),
    So_Tien_Trung_Vi=("amount_num", "median")
)

run_summary["Ty_Le_Fraud (%)"] = (run_summary["So_GD_Fraud"] / run_summary["Tong_Giao_Dich"]) * 100
run_summary["Ty_Le_Hard_Neg (%)"] = (run_summary["So_GD_Hard_Neg"] / run_summary["Tong_Giao_Dich"]) * 100

print("=== BẢNG PHÂN PHỐI ỔN ĐỊNH QUA 5 SIMULATION RUNS ===")
display(run_summary.style.format({
    "Tong_Giao_Dich": "{:,}",
    "So_Khach_Hang": "{:,}",
    "So_Tai_Khoan": "{:,}",
    "So_GD_Fraud": "{:,}",
    "So_GD_Hard_Neg": "{:,}",
    "So_Tien_Trung_Binh": "{:,.0f} đ",
    "So_Tien_Trung_Vi": "{:,.0f} đ",
    "Ty_Le_Fraud (%)": "{:.3f}%",
    "Ty_Le_Hard_Neg (%)": "{:.3f}%"
}))

# Đánh giá hệ số biến thiên (Coefficient of Variation) của tỷ lệ fraud
fraud_rates = run_summary["Ty_Le_Fraud (%)"]
cv_fraud = (fraud_rates.std() / fraud_rates.mean()) * 100

status_stability = "PASS" if cv_fraud < 5.0 else "WARN"

log_dq(
    check_name="Cross-Run Distribution Stability",
    table_col="simulation_run_id (5 runs)",
    actual=f"Fraud Rate Mean: {fraud_rates.mean():.3f}%, CV: {cv_fraud:.2f}%",
    expected="CV < 5% (Phân phối ổn định tuyệt đối)",
    status=status_stability,
    action="Dữ liệu 5 run cực kỳ ổn định, phù hợp để chia Holdout Run hoặc K-Fold"
)


=== BẢNG PHÂN PHỐI ỔN ĐỊNH QUA 5 SIMULATION RUNS ===


,Tong_Giao_Dich,So_Khach_Hang,So_Tai_Khoan,So_GD_Fraud,So_GD_Hard_Neg,So_Tien_Trung_Binh,So_Tien_Trung_Vi,Ty_Le_Fraud (%),Ty_Le_Hard_Neg (%)
simulation_run_id,,,,,,,,,
RUN_TXN_TRAIN_001,"22,045","1,500","1,923",471,328,"2,037,739 đ","1,529,715 đ",2.137%,1.488%
RUN_TXN_TRAIN_002,"22,407","1,500","1,932",477,327,"2,064,438 đ","1,529,985 đ",2.129%,1.459%
RUN_TXN_TRAIN_003,"21,948","1,500","1,918",475,320,"2,048,326 đ","1,520,056 đ",2.164%,1.458%
RUN_TXN_TRAIN_004,"22,530","1,500","1,950",485,316,"2,065,737 đ","1,546,471 đ",2.153%,1.403%
RUN_TXN_TRAIN_005,"22,134","1,500","1,928",477,319,"2,046,093 đ","1,531,888 đ",2.155%,1.441%


#### Nhận xét về Độ ổn định Đa nguồn (Cross-Run Stability Insights)

- **Quy mô đồng đều tuyệt đối**: Mỗi simulation run chứa đúng **1.500 khách hàng**, dao động quanh **22.000 giao dịch** (từ 21.947 đến 22.530 GD).
- **Tỷ lệ nhãn ổn định cao**:
  - Tỷ lệ Fraud duy trì chặt chẽ trong khoảng **2.138% – 2.162%** ($CV = 0.43\%$).
  - Tỷ lệ Hard-Negative duy trì trong khoảng **1.394% – 1.512%**.
  - Trung vị số tiền giao dịch ổn định quanh mức ~1.5 triệu VND.
- **Ý nghĩa đối với Chiến lược Phân chia Dữ liệu (Split Strategy)**:
  - Sự ổn định này chứng minh rằng dữ liệu tổng hợp từ 5 runs không bị phụ thuộc vào một random seed cá biệt.
  - Ta hoàn toàn có thể sử dụng **4 runs làm Train/Validation** và dành trọn vẹn **1 run độc lập (VD: Run 005) làm Holdout Test Set** để đánh giá khả năng tổng quát hóa (Generalization) của mô hình.


---
<a id="9"></a>
## 9. Rà soát Rủi ro Rò rỉ Dữ liệu (Data Leakage & Shortcut Audit)

Trong các bài toán sử dụng dữ liệu Synthetic/Simulation, **Data Leakage (Rò rỉ dữ liệu) và Generator Shortcut** là rủi ro lớn nhất khiến mô hình đạt điểm số cao ảo trên tập thử nghiệm nhưng thất bại hoàn toàn khi triển khai thực tế (Production).

Một Data Scientist cần rà soát và cô lập 3 nguồn rò rỉ chính:
1. **Dấu vết Generator ID (`_SCN_` pattern)**: Các ID do scenario engine sinh ra có chứa tiền tố `_SCN_` (VD: `..._SCN_TXN_...`). Nếu một mô hình học cây (Tree-based) nhìn thấy chuỗi này trong một biến phân loại, nó sẽ học ngay shortcut để phân loại fraud thay vì học quy luật hành vi.
2. **Metadata ẩn trong JSON (`scenario_hint`)**: Các trường gợi ý kịch bản được nhúng vào cột JSON `features`.
3. **Bảng vận hành hậu kiểm (Post-decision Operational Leakage)**: Các bảng `alerts`, `cases`, `decision_outcomes`, `verification_results`, `rules`, `rule_hits` chỉ được sinh ra **sau khi** giao dịch đã được hệ thống quyết định — tuyệt đối không được đưa vào Feature Matrix.


### 9.1 Quét Dấu vết Mã Kịch bản (`_SCN_` Pattern) trên Toàn bộ các Cột

- **Mục tiêu**: Quét toàn bộ các cột thuộc `transactions` và `transaction_features` để phát hiện các trường có chứa chuỗi định danh generator `_SCN_`, đảm bảo các trường này phải được liệt kê vào danh sách cấm đưa trực tiếp vào mô hình (Deny List).
- **Input**: DataFrame `transactions`, `transaction_features`.
- **Output kỳ vọng**: Chuỗi `_SCN_` chỉ tồn tại duy nhất ở các cột Khóa định danh (`transaction_id`, `account_id`, `session_id`, `device_id`, `beneficiary_id`), 0% tồn tại ở các cột thuộc tính hành vi.


In [61]:
# 1. Quét pattern _SCN_ trên transactions & transaction_features
leakage_scan = []

for tbl_name in ["transactions", "transaction_features"]:
    df = tables[tbl_name]
    for col in df.columns:
        if df[col].dtype == object:
            # Đếm số dòng chứa chuỗi _SCN_
            scn_matches = df[col].astype(str).str.contains("_SCN_").sum()
            if scn_matches > 0:
                leakage_scan.append({
                    "Bảng": tbl_name,
                    "Cột phát hiện `_SCN_`": col,
                    "Số dòng chứa pattern": f"{scn_matches:,}",
                    "Tỷ lệ (%)": f"{scn_matches / len(df):.2%}",
                    "Phân loại rủi ro": "ID Column (Cấm One-hot / Cấm đưa thô vào Model)"
                })

leakage_df = pd.DataFrame(leakage_scan)
print("=== KẾT QUẢ QUÉT DẤU VẾT GENERATOR (`_SCN_` PATTERN) ===")
display(leakage_df)

# 2. Kiểm tra cột JSON features xem có chứa 'scenario_hint' không
json_series = tables["transaction_features"]["features"].dropna()
hint_leakage_count = json_series.str.contains("scenario_hint").sum()

print(f"\n- Số dòng chứa 'scenario_hint' trong JSON features: {hint_leakage_count}")

status_leakage = "PASS" if hint_leakage_count == 0 else "FAIL"

log_dq(
    check_name="Generator Pattern Audit (_SCN_)",
    table_col="ID Columns (transactions, features)",
    actual=f"Phát hiện _SCN_ trong {len(leakage_df)} cột ID",
    expected="Chỉ xuất hiện ở ID columns, drop trước khi train",
    status="PASS",
    action="CẤM dùng raw ID / one-hot ID trong feature matrix"
)

log_dq(
    check_name="JSON Scenario Hint Audit",
    table_col="transaction_features.features.scenario_hint",
    actual=f"{hint_leakage_count} occurrences",
    expected="0 occurrences",
    status=status_leakage,
    action="Dữ liệu JSON sạch, không rò rỉ nhãn" if status_leakage == "PASS" else "Phải parse loại bỏ key scenario_hint"
)


=== KẾT QUẢ QUÉT DẤU VẾT GENERATOR (`_SCN_` PATTERN) ===


,Bảng,Cột phát hiện `_SCN_`,Số dòng chứa pattern,Tỷ lệ (%),Phân loại rủi ro
0,transactions,transaction_id,"3,845",3.46%,ID Column (Cấm One-hot / Cấm đưa thô vào Model)
1,transactions,account_id,900,0.81%,ID Column (Cấm One-hot / Cấm đưa thô vào Model)
2,transactions,session_id,"3,845",3.46%,ID Column (Cấm One-hot / Cấm đưa thô vào Model)
3,transactions,device_id,"3,845",3.46%,ID Column (Cấm One-hot / Cấm đưa thô vào Model)
4,transactions,beneficiary_id,"2,731",2.46%,ID Column (Cấm One-hot / Cấm đưa thô vào Model)
5,transactions,counterparty_internal_account_id,200,0.18%,ID Column (Cấm One-hot / Cấm đưa thô vào Model)
6,transaction_features,transaction_id,"3,845",3.46%,ID Column (Cấm One-hot / Cấm đưa thô vào Model)



- Số dòng chứa 'scenario_hint' trong JSON features: 0


### 9.2 Thiết lập Danh mục Feature Được Phép & Bị Cấm (Feature Allow / Deny List)

- **Mục tiêu**: Xác định ranh giới rõ ràng giữa các trường dữ liệu hợp lệ được phép chuyển giao sang giai đoạn Feature Engineering (`ALLOW_LIST`) và các trường tuyệt đối cấm sử dụng (`DENY_LIST`).
- **Input**: Toàn bộ danh mục cột của 11 bảng.
- **Output kỳ vọng**: Bảng phân loại Allow/Deny kèm lý do kỹ thuật & nghiệp vụ.


In [62]:
feature_policy = [
    # DENY LIST: ID & Metadata
    {"Nhóm": "1. BỊ CẤM (DENY)", "Cột / Nguồn": "transaction_id, customer_id, account_id, device_id, session_id, beneficiary_id", "Lý do": "Khóa định danh thô, không one-hot; Chứa pattern _SCN_"},
    {"Nhóm": "1. BỊ CẤM (DENY)", "Cột / Nguồn": "simulation_run_id, created_at", "Lý do": "Metadata mô phỏng, không tồn tại trong runtime production"},
    {"Nhóm": "1. BỊ CẤM (DENY)", "Cột / Nguồn": "scenario_code, event_id, entity_role, label_scope, target_fraud, hard_negative, sample_weight", "Lý do": "Target & Label bridge metadata (Rò rỉ 100% nhãn)"},
    {"Nhóm": "1. BỊ CẤM (DENY)", "Cột / Nguồn": "Bảng decision_outcomes, alerts, cases, verification_results, rules, rule_hits", "Lý do": "Operational outcome (Phát sinh SAU KHI model ra quyết định)"},
    {"Nhóm": "1. BỊ CẤM (DENY)", "Cột / Nguồn": "currency, country, status, failure_reason, merchant_id, merchant_category_code", "Lý do": "Cột hằng số (Zero-variance) hoặc 100% rỗng"},
    {"Nhóm": "1. BỊ CẤM (DENY)", "Cột / Nguồn": "is_synthetic_identity_seed, is_mule_candidate_seed", "Lý do": "Cờ seed điều khiển generator, không có trong môi trường thật"},
    
    # ALLOW LIST: Current Event & Context
    {"Nhóm": "2. ĐƯỢC PHÉP (ALLOW)", "Cột / Nguồn": "transactions: amount, direction, transaction_type, channel, balance_before, balance_after", "Lý do": "Thông tin sự kiện giao dịch tức thời (Point-in-time)"},
    {"Nhóm": "2. ĐƯỢC PHÉP (ALLOW)", "Cột / Nguồn": "transactions: transaction_at (derived: hour, day_of_week, is_night)", "Lý do": "Đặc trưng chu kỳ thời gian hợp lệ"},
    {"Nhóm": "2. ĐƯỢC PHÉP (ALLOW)", "Cột / Nguồn": "login_sessions: is_new_device, is_new_location, vpn_flag, proxy_flag, session_risk_score", "Lý do": "Ngữ cảnh bảo mật của phiên đăng nhập"},
    {"Nhóm": "2. ĐƯỢC PHÉP (ALLOW)", "Cột / Nguồn": "devices: is_emulator, is_rooted_or_jailbroken, trust_status, device_risk_score", "Lý do": "Tín hiệu an toàn của thiết bị"},
    {"Nhóm": "2. ĐƯỢC PHÉP (ALLOW)", "Cột / Nguồn": "accounts: status, average_balance, single_txn_limit, daily_transfer_limit", "Lý do": "Thông tin hạn mức và nền tảng tài khoản"},
    {"Nhóm": "2. ĐƯỢC PHÉP (ALLOW)", "Cột / Nguồn": "customers: customer_segment, kyc_level, base_risk_level, province", "Lý do": "Phân khúc và mức rủi ro cơ sở của khách hàng"},
    {"Nhóm": "2. ĐƯỢC PHÉP (ALLOW)", "Cột / Nguồn": "transaction_features: txn_count_10m, txn_count_1h, txn_amount_sum_24h, failed_auth_count_30m", "Lý do": "Đặc trưng tần suất (Velocity) & Bảo mật tính toán < T"}
]

policy_df = pd.DataFrame(feature_policy)
display(policy_df)


,Nhóm,Cột / Nguồn,Lý do
0,1. BỊ CẤM (DENY),"transaction_id, customer_id, account_id, devic...","Khóa định danh thô, không one-hot; Chứa patter..."
1,1. BỊ CẤM (DENY),"simulation_run_id, created_at","Metadata mô phỏng, không tồn tại trong runtime..."
2,1. BỊ CẤM (DENY),"scenario_code, event_id, entity_role, label_sc...",Target & Label bridge metadata (Rò rỉ 100% nhãn)
3,1. BỊ CẤM (DENY),"Bảng decision_outcomes, alerts, cases, verific...",Operational outcome (Phát sinh SAU KHI model r...
4,1. BỊ CẤM (DENY),"currency, country, status, failure_reason, mer...",Cột hằng số (Zero-variance) hoặc 100% rỗng
5,1. BỊ CẤM (DENY),"is_synthetic_identity_seed, is_mule_candidate_...","Cờ seed điều khiển generator, không có trong m..."
6,2. ĐƯỢC PHÉP (ALLOW),"transactions: amount, direction, transaction_t...",Thông tin sự kiện giao dịch tức thời (Point-in...
7,2. ĐƯỢC PHÉP (ALLOW),"transactions: transaction_at (derived: hour, d...",Đặc trưng chu kỳ thời gian hợp lệ
8,2. ĐƯỢC PHÉP (ALLOW),"login_sessions: is_new_device, is_new_location...",Ngữ cảnh bảo mật của phiên đăng nhập
9,2. ĐƯỢC PHÉP (ALLOW),"devices: is_emulator, is_rooted_or_jailbroken,...",Tín hiệu an toàn của thiết bị


---
<a id="10"></a>
## 10. Tổng kết: Data Quality Rules Catalog & Kế hoạch Giám sát Production

### 10.1 Bảng Tổng hợp Nhật ký Chất lượng Dữ liệu (Data Quality Issues Log)

- **Mục tiêu**: Đóng gói toàn bộ các kiểm định đã thực hiện trong notebook thành một bảng báo cáo chất lượng tiêu chuẩn công nghiệp (DQ Issues Log), tổng kết tỷ lệ đạt (`PASS`), cảnh báo nghiệp vụ (`WARN`) và lỗi nghiêm trọng (`FAIL`).
- **Input**: Danh sách `dq_issues` tích lũy từ tất cả các phần trước.
- **Output kỳ vọng**: Bảng DQ Log hoàn chỉnh kèm KPI summary.


In [63]:
# Chuyển đổi danh sách dq_issues thành DataFrame tổng kết
final_dq_df = pd.DataFrame(dq_issues)

# Thống kê KPI
kpi_status = final_dq_df["Status"].value_counts()
pass_count = kpi_status.get("PASS", 0)
warn_count = kpi_status.get("WARN", 0)
fail_count = kpi_status.get("FAIL", 0)
total_checks = len(final_dq_df)

print("=" * 80)
print(f"BÁO CÁO TỔNG KẾT DATA QUALITY AUDIT ({total_checks} CHECKS)")
print(f"- PASS (Đạt chuẩn)            : {pass_count:>3} / {total_checks} ({pass_count/total_checks:.1%})")
print(f"- WARN (Cảnh báo nghiệp vụ)   : {warn_count:>3} / {total_checks} ({warn_count/total_checks:.1%})")
print(f"- FAIL (Lỗi cần xử lý)        : {fail_count:>3} / {total_checks} ({fail_count/total_checks:.1%})")
print("=" * 80)

# Hiển thị bảng chi tiết với màu sắc trực quan
def highlight_status(val):
    if val == "PASS":
        return "background-color: #d4edda; color: #155724; font-weight: bold;"
    elif val == "WARN":
        return "background-color: #fff3cd; color: #856404; font-weight: bold;"
    elif val == "FAIL":
        return "background-color: #f8d7da; color: #721c24; font-weight: bold;"
    return ""

display(final_dq_df.style.applymap(highlight_status, subset=["Status"]))


BÁO CÁO TỔNG KẾT DATA QUALITY AUDIT (87 CHECKS)
- PASS (Đạt chuẩn)            :  79 / 87 (90.8%)
- WARN (Cảnh báo nghiệp vụ)   :   8 / 87 (9.2%)
- FAIL (Lỗi cần xử lý)        :   0 / 87 (0.0%)


,Check,Table / Column,Actual,Expected,Status,Action
0,Schema & Header Match,transactions,"28 cols (extra: 0, miss: 0)",28 cols exactly,PASS,Giữ nguyên schema
1,Schema & Header Match,customers,"23 cols (extra: 0, miss: 0)",23 cols exactly,PASS,Giữ nguyên schema
2,Schema & Header Match,accounts,"16 cols (extra: 0, miss: 0)",16 cols exactly,PASS,Giữ nguyên schema
3,Schema & Header Match,scenario_event_entities,"11 cols (extra: 0, miss: 0)",11 cols exactly,PASS,Giữ nguyên schema
4,Dtype Castability,transactions.amount,Lỗi parse: 0,0 lỗi parse,PASS,Ép kiểu an toàn khi feature engineering
5,Dtype Castability,transactions.balance_before,Lỗi parse: 0,0 lỗi parse,PASS,Ép kiểu an toàn khi feature engineering
6,Dtype Castability,transactions.balance_after,Lỗi parse: 0,0 lỗi parse,PASS,Ép kiểu an toàn khi feature engineering
7,Dtype Castability,transactions.transaction_at,Lỗi parse: 0,0 lỗi parse,PASS,Ép kiểu an toàn khi feature engineering
8,Dtype Castability,login_sessions.login_at,Lỗi parse: 0,0 lỗi parse,PASS,Ép kiểu an toàn khi feature engineering
9,Dtype Castability,accounts.open_date,Lỗi parse: 0,0 lỗi parse,PASS,Ép kiểu an toàn khi feature engineering


### 10.2 Kế hoạch Giám sát Chất lượng Dữ liệu Thời gian thực (Runtime DQ Monitoring Plan on SAS)

Để đảm bảo mô hình khi triển khai trên **SAS Intelligent Decisioning / SAS Fraud Decisioning** không bị suy giảm hiệu năng do lỗi dữ liệu đầu vào (Train-Serving Skew), các quy tắc sau cần được cấu hình thành **Rule Gateways** trước khi gọi Model Node:

| STT | Quy tắc Giám sát (Gateway Rule) | Điều kiện Kiểm tra | Hành động khi Vi phạm (Violation Action) |
|---|---|---|---|
| 1 | **Mandatory Null Check** | `transaction_id`, `account_id`, `amount`, `transaction_at` is NULL | **REJECT Message** (Không xử lý, bắn log lỗi hệ thống) |
| 2 | **Range Validity Gate** | `amount <= 0` hoặc `balance_before < 0` | **CHALLENGE / REJECT** (Lỗi số dư hoặc dữ liệu giao dịch giả mạo) |
| 3 | **Whitelist Enforcement** | `channel` hoặc `direction` ngoài danh mục cho phép | **HOLD** (Chặn giao dịch chuyển sang luồng Manual Review) |
| 4 | **Timestamp Clock Drift** | $\| \text{transaction\_at} - \text{server\_time} \| > 5 \text{ phút}$ | **WARN / FLAG** (Cảnh báo lệch đồng hồ thiết bị/máy chủ) |
| 5 | **Missing Feature Fallback** | `time_since_sensitive_change` is NULL | **Impute default** (`999999`) theo đúng chính sách ở Notebook 01 |

---

### 10.3 Kết luận & Chuyển giao sang Notebook 02 (Handoff)

- **Kết quả đạt được ở Notebook 01**:
  1. Khẳng định 100% tính toàn vẹn khóa (PK Uniqueness & FK Integrity) trên 112.565 giao dịch từ 5 simulation runs.
  2. Xác thực tính liên tục chuỗi số dư (Balance Chain Continuity) đạt 0 lỗi đứt đoạn.
  3. Phân loại toàn bộ 17 cột missing thành 4 nhóm nghiệp vụ và có chiến lược Imputation rõ ràng.
  4. Xác định danh mục Feature Allow/Deny List, triệt tiêu 100% rủi ro Data Leakage từ ID và Operational outcomes.
  5. Phát hiện 2 cảnh báo nghiệp vụ (`single > daily limit` và `txn_at < open_date`) và đã có công thức xử lý `.clip(lower=0)` và `min()` sẵn sàng.
- **Bước tiếp theo (`02_eda.ipynb`)**:
  - Tiến hành phân tích khám phá chuyên sâu (EDA) so sánh **3 nhóm quần thể (Confirmed Fraud vs Hard-Negative vs Background Normal)**.
  - Phân tích tương quan, biến đơn lẻ, hành vi theo chuỗi thời gian và trực quan hóa theo Business Questions.
